### Links and Resources
| Function | Use |
|---|---|
| [`struct`, `getField`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.struct.html) | Build / read named tuples |
| [`map`, `create_map`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.create_map.html) | Build maps |
| [`map_keys`, `map_values`, `map_entries`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.map_keys.html) | Map projection |
| [`from_json`, `to_json`, `schema_of_json`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html) | Parse / emit JSON |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Map, Struct, and JSON Functions

The "complex types" tier of Spark's type system:

- **struct** — fixed schema, named fields. Like a row inside a row.
- **map** — open schema, key→value, all keys same type. Like a Python dict.
- **JSON strings** — neither, until you parse them.

Knowing when to reach for which is the main skill here. Rule of thumb: if you know the field names at design time, use a **struct**. If keys are data, use a **map**.

## Structs — build with `struct`, read with `.field` or `getField`

In [ ]:
df = spark.createDataFrame([
    (1, "Alice", "alice@x.io",  85.0),
    (2, "Bob",   "bob@x.io",    72.5),
], ["id", "name", "email", "score"])

with_struct = df.withColumn("contact", F.struct("name", "email"))
with_struct.printSchema()
with_struct.show(truncate=False)

# Reading back
with_struct.select(
    "id",
    F.col("contact.name").alias("name"),
    F.col("contact").getField("email").alias("email"),
).show(truncate=False)

## Modifying structs — `withField`, `dropFields` (Spark 3.1+)

The most common reason people regret using a struct is that adding a field used to mean rebuilding the whole thing. No longer.

In [ ]:
updated = with_struct.withColumn(
    "contact",
    F.col("contact").withField("verified", F.lit(True)).dropFields("email"),
)
updated.printSchema()
updated.select("contact").show(truncate=False)

## Maps — `create_map`, `map_keys`, `map_values`, `map_entries`

`create_map` takes alternating key/value column expressions.

In [ ]:
m = df.withColumn(
    "props",
    F.create_map(
        F.lit("name"),  F.col("name"),
        F.lit("score"), F.col("score").cast("string"),
    ),
)
m.printSchema()
m.show(truncate=False)

m.select(
    F.col("props")["name"].alias("name_via_map"),
    F.map_keys("props").alias("keys"),
    F.map_values("props").alias("values"),
).show(truncate=False)

## When to pick map vs struct

| Use a **struct** when… | Use a **map** when… |
|---|---|
| Keys are known at design time | Keys are data (variable per row) |
| You want per-field types (`int`, `string`, `array`...) | All values share a type |
| You want column-style autocomplete & schema enforcement | You're modeling sparse / extensible properties |
| You'll write SQL like `WHERE contact.email LIKE ...` | You'll iterate over keys |

## JSON ingestion — `from_json` with a schema

Almost every Kafka/Auto Loader pipeline does this. **Always pass an explicit schema**: relying on schema inference blows up cost and yields surprising types.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

raw = spark.createDataFrame([
    (1, '{"name": "Alice", "score": 85.0, "tags": ["a", "b"]}'),
    (2, '{"name": "Bob",   "score": 72.5, "tags": []}'),
    (3, '{"name": "Eve",   "score": null, "tags": null}'),
    (4, '{not valid json}'),  # malformed → all fields NULL
], ["id", "payload"])

schema = StructType([
    StructField("name",  StringType()),
    StructField("score", DoubleType()),
    StructField("tags",  ArrayType(StringType())),
])

parsed = raw.withColumn("p", F.from_json("payload", schema))
parsed.select("id", "p.name", "p.score", "p.tags").show(truncate=False)

## Inferring a schema once with `schema_of_json`

If you really don't know the shape, sample one row, infer the schema, then *reuse* it. Don't infer per-row.

In [ ]:
sample = '{"name": "Alice", "score": 85.0, "tags": ["a"]}'

inferred = spark.range(1).select(F.schema_of_json(F.lit(sample))).collect()[0][0]
print("inferred:", inferred)

# Then plug it in
raw.withColumn("p", F.from_json("payload", inferred)).select("id", "p.*").show()

## `to_json` — serialize back

Useful for emitting to Kafka or for compact storage of variable-shape data.

In [ ]:
with_struct.select(
    "id",
    F.to_json("contact").alias("contact_json"),
).show(truncate=False)

## `get_json_object` and `json_tuple` — for the lazy path

If you only want one or two fields out of a deeply nested JSON, you can skip the schema:

- `get_json_object(col, "$.path.to.field")` — JSONPath, returns string.
- `json_tuple(col, "k1", "k2", ...)` — pulls multiple top-level keys at once.

In [ ]:
raw.select(
    "id",
    F.get_json_object("payload", "$.name").alias("name"),
    F.get_json_object("payload", "$.tags[0]").alias("first_tag"),
).show(truncate=False)

## Takeaways

- **Struct** when keys are fixed; **map** when keys are data.
- `withField` / `dropFields` make struct evolution painless.
- `from_json` with an **explicit schema** is the right way to parse JSON columns at scale.
- Use `schema_of_json` once on a sample, reuse the result — don't infer per row.
- `get_json_object` is a fine shortcut for a few fields, but the typed `from_json` path is faster and safer for production.